# 03. Carga politécnica disponible

**Fuente:** `data/raw/cargapolitecnicadisponible.csv`  
**Salida:** `data/processed/carga_politecnica_disponible.csv`

Registra las responsabilidades de gestión institucional, cargos administrativos, comisiones y actividades no docentes asignadas al personal en cada período académico (horas dedicadas, tipo de función, unidad académica u operativa). Aporta variables clave sobre liderazgo, gestión y servicio politécnico.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import pandas as pd
import _preprocesamiento_comun as pc

pd.set_option('display.max_columns', 100)

In [ ]:
df = pc.leer_csv('cargapolitecnicadisponible.csv', low_memory=False)
df.head()

## 2. Exploración inicial

In [ ]:
pc.resumen(df, 'carga_politecnica_disponible')

In [ ]:
# 1. Normalizar las columnas de términos

columnas_terminos = ['TERMINO1', 'TERMINO2', 'TERMINO3']

for col in columnas_terminos:
    df[col] = ( df[col].fillna('').astype(str).str.strip())

In [ ]:
# 2. Crear indicadores de aplicación por término

df['APLICA_T1'] = (df['TERMINO1'] != '').astype(int)
df['APLICA_T2'] = (df['TERMINO2'] != '').astype(int)
df['APLICA_T3'] = (df['TERMINO3'] != '').astype(int)

In [ ]:
# 3. Número de términos en los que aplica la actividad

df['NUM_TERMINOS'] = ( df['APLICA_T1'] + df['APLICA_T2'] + df['APLICA_T3'])

In [ ]:
# 4. Crear una representación legible de los términos

def consolidar_terminos(row):
    activos = []
    if row['APLICA_T1'] == 1:
        activos.append('1')
    if row['APLICA_T2'] == 1:
        activos.append('2')
    if row['APLICA_T3'] == 1:
        activos.append('3')
    return '-'.join(activos) if activos else 'SIN_TERMINO'


df['TERMINOS_ACTIVOS'] = df.apply( consolidar_terminos, axis=1)


In [ ]:
df = df.drop( columns=['TERMINO1', 'TERMINO2', 'TERMINO3'], errors='ignore'
)

In [ ]:
pc.resumen(df, 'carga_politecnica_disponible')

In [ ]:
df.dtypes

## 3. Limpieza

In [ ]:
df = pc.limpiar_strings(df)
df = pc.quitar_columnas_vacias(df, umbral=0.99)
df = pc.quitar_columnas_constantes(df, excluir=["TERMINON1", "TERMINON2", "TERMINON3"])

In [ ]:
df = df.drop(columns=['EMAILALTERNO'], errors='ignore')
df = df.drop(columns=['EMAIL'], errors='ignore')

## 5. Tipado de fechas e identificadores

In [ ]:
df = pc.castear_fechas(df, ['FECHAAPROBPLANIF', 'FECHAINICIOREGIST', 'FECHA_INICIO', 'FECHA_FIN'])
df = pc.castear_enteros(df, ['IDPERSONA', 'IDCURSO', 'IDUNIDAD', 'IDUNIDADCARGA', 'IDMATERIA', 'IDTIPOACTIVIDAD', 'IDPERIODO', 'IDCPLCAMBIOSCURSO'])
df = pc.convertir_sn_a_binario(df, ['APROBADO'])

## Gráficos exploratorios

Vistas rápidas para apoyar la construcción del catálogo de variables del perfil (Fase 1-2 de la metodología): estacionalidad/tendencia temporal, categorías dominantes y forma de la distribución de las variables numéricas.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (9, 4)

**Actividades de carga politécnica por año**.

In [ ]:
conteo_anio = df['ANIO'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(9, 4))
conteo_anio.plot(kind='bar', ax=ax, color=pc.COLOR_PRINCIPAL)
ax.set_title('Actividades de carga politécnica por año')
ax.set_xlabel('Año'); ax.set_ylabel('N.º de registros')
plt.tight_layout()

**Unidades con más actividades de carga politécnica** (gestión, comisiones, direcciones).

In [ ]:
pc.grafico_barras(df['NOMBREUNIDAD'], 'Top 10 unidades por actividades de carga politécnica', top=10)

**Número de términos en los que aplica cada actividad** (variable derivada NUM_TERMINOS).

In [ ]:
conteo_terminos = df['NUM_TERMINOS'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
conteo_terminos.plot(kind='bar', ax=ax, color=pc.COLOR_PRINCIPAL)
ax.set_title('N.º de términos en los que aplica la actividad')
ax.set_xlabel('NUM_TERMINOS'); ax.set_ylabel('N.º de registros')
plt.tight_layout()

## 7. Verificación final

In [ ]:
pc.resumen(df, 'carga_politecnica_disponible (procesado)')
df.head()

## 8. Guardado en data/processed

In [ ]:
pc.guardar_procesado(df, 'carga_politecnica_disponible.csv')